# How I understood TensorFlow using what I already knew from ArcGIS

To practice, I adapted the TensorFlow beginner tutorial, which uses handwritten digits (MNIST), to satellite images. 
I used EuroSAT, a dataset of 27,000 Sentinel-2 images. Each image is 64 × 64 pixels, with 3 bands (red, green, and blue), 
and belongs to one of 10 land-cover classes, such as Forest, River, Highway, or AnnualCrop. I followed the same five steps 
as the tutorial: load the data, build the model, compile it, train it, and evaluate it.

### 1. Set up TensorFlow


In [11]:
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

###  2. Load the data

##### Import the libraries

In [12]:
import os
import zipfile
import urllib.request

#### Download the zip file and unzip it

In [13]:
url = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1"
folder = "eurosat_data"
zip_path = os.path.join(folder, "EuroSAT_RGB.zip")

os.makedirs(folder, exist_ok=True)

if not os.path.exists(zip_path):
    print("Downloading EuroSAT (about 95 MB)...")
    urllib.request.urlretrieve(url, zip_path)
    print("Download finished.")

with zipfile.ZipFile(zip_path) as z:
    z.extractall(folder)

print("Inside the folder:", os.listdir(folder))

Inside the folder: ['EuroSAT_RGB', 'EuroSAT_RGB.zip']


#### Load the images into TensorFlow

##### Keras reads the 10 subfolders and uses their names as the classes. 80% of the images are used for training and 20% for testing, in batches of 32.

In [14]:
data_dir = "eurosat_data/EuroSAT_RGB"

train_ds, test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=42,
    image_size=(64, 64),
    batch_size=32,
)

print("Classes:", train_ds.class_names)

Found 27000 files belonging to 10 classes.
Using 21600 files for training.
Using 5400 files for validation.
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


Keras splits the images into 80% for training (21,600 images) and 20% for testing (5,400 images). 
It also groups them into batches of 32 images, because the model learns from a few images at a time. 
One batch has the shape (32, 64, 64, 3): 32 images, 64 rows, 64 columns, and 3 bands. For a GIS user, a tensor is a stack of rasters.

### 3. Build the model

The Input layer describes the data: an image of 64 rows, 64 columns, and 3 bands. The Rescaling layer divides every pixel value by 255, so the values go from 0 to 1, like dividing a raster in Raster Calculator. The Flatten layer unrolls the image into one line of 12,288 numbers (64 × 64 × 3), like exporting every cell of a raster into one column of a table. The first Dense layer has 128 neurons. Each neuron looks at all these numbers and learns to detect a different pattern, such as green areas or gray lines. The ReLU activation then sets negative values to zero, like a reclassification. The Dropout layer switches off 20% of these neurons at random during training, so the model does not memorize the training images. The last Dense layer gives 10 scores, one for each land-cover class, and the class with the highest score is the model’s answer.

When I looked at my code, I could not find the weights anywhere. Keras creates the weights inside the Dense layers. When I write Dense(128), Keras creates the weights this layer needs, gives them random values, and manages them. To understand what a weight is, I thought of the Weighted Overlay tool in ArcGIS. In this tool, I give each input layer a weight, for example 40% for slope, 35% for land cover, and 25% for distance to roads. These weights decide how much each input counts in the result. A Dense layer works the same way, but the model starts with random weights and adjusts them during training.

In [15]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(64, 64, 3)),               # input: 64 rows, 64 columns, 3 bands (Red, Green, Blue)
    tf.keras.layers.Rescaling(1.0 / 255),            # pixel values from 0-255 to 0-1
    tf.keras.layers.Flatten(),                       # 64 x 64 x 3 block -> one line of 12,288 numbers
    tf.keras.layers.Dense(128, activation="relu"),   # 128 neurons find patterns; relu sets negative values to 0
    tf.keras.layers.Dropout(0.2),                    # switches off 20% of the neurons during training
    tf.keras.layers.Dense(10),                       # 10 scores, one per land-cover class
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)              │ (None, 64, 64, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 12288)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │       1,572,992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,574,282 (6.01 MB)

 Trainable params: 1,574,282 (6.01 MB)

 Non-trainable params: 0 (0.00 B)

###  4. Train the model

The loss function measures how wrong the model is: the lower the loss, the better. The optimizer (Adam) adjusts the weights after each batch, and the learning rate sets the size of each adjustment.

With the default learning rate (0.001), my first training stayed at 10% accuracy, the same as guessing among 10 classes: the model gave the same answer for every image. With a learning rate 10 times smaller (0.0001), the model learned.

In [16]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer= tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss=loss_fn,
              metrics=["accuracy"]) # accuracy shows the percentage of correct answers

history = model.fit(train_ds, epochs=20)

Epoch 1/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.2527 - loss: 1.9538
Epoch 2/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3082 - loss: 1.7907
Epoch 3/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3308 - loss: 1.7303
Epoch 4/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3499 - loss: 1.6970
Epoch 5/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3644 - loss: 1.6693
Epoch 6/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3813 - loss: 1.6475
Epoch 7/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3877 - loss: 1.6315
Epoch 8/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3944 - loss: 1.6116
Epoch 9/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4050 - loss: 1.6003
Epoch 10/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4100 - loss: 1.5854
Epoch 11/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4140 - loss: 1.5769
Epoch 12/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/st

### 5. Evaluate the model

In [17]:
model.evaluate(test_ds, verbose=2)

probability_model = tf.keras.Sequential([
    model,
    tf.keras.layers.Softmax()
])

class_names = train_ds.class_names

for images, labels in test_ds.take(1):
    probabilities = probability_model(images[:5]).numpy()

for i in range(5):
    predicted = class_names[probabilities[i].argmax()]
    correct = class_names[int(labels[i])]
    confidence = probabilities[i].max()
    print(f"Image {i+1}: predicted {predicted} ({confidence:.0%}), correct {correct}")

169/169 - 3s - 18ms/step - accuracy: 0.4693 - loss: 1.4436
Image 1: predicted Pasture (41%), correct Pasture
Image 2: predicted AnnualCrop (42%), correct AnnualCrop
Image 3: predicted AnnualCrop (23%), correct Highway
Image 4: predicted SeaLake (95%), correct SeaLake
Image 5: predicted AnnualCrop (37%), correct AnnualCrop


On the 5,400 test images, the model reached **47% accuracy** (loss 1.44). With 10 classes, guessing would give 10%.

In this sample of five test images, the model found four correct classes. It was sure about the lake (95%), because water is a large, uniform area. It was wrong about the highway, which it classified as AnnualCrop.

**Why only 47%?** The Flatten layer turns the image into 12,288 separate numbers, so the model cannot see shapes, like a thin road crossing fields. Each pixel position also needs its own weights. The vegetation classes look very similar in RGB at 10 m, and the accuracy was still going up at epoch 20. Convolution layers solve the first two problems. That is the next notebook.

### Links

- Article on Medium: [link]
- TensorFlow 2 quickstart for beginners: https://www.tensorflow.org/tutorials/quickstart/beginner
- EuroSAT: Helber, P., Bischke, B., Dengel, A., & Borth, D. (2019). EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification. *IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing*, 12(7), 2217–2226. Data: https://zenodo.org/records/7711810